### Import the Libraries

In [13]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

StatementMeta(, 7f236da4-c644-4002-809a-9f04bdba72ec, 29, Finished, Available, Finished, False)

### Read Silver data

In [14]:
silver_df=spark.read.format('delta').load("abfss://Kiran@onelake.dfs.fabric.microsoft.com/RT_Project.Lakehouse/Tables/Silver/api_silver_data")

StatementMeta(, 7f236da4-c644-4002-809a-9f04bdba72ec, 30, Finished, Available, Finished, False)

### Deduplicated Customer Dataframe creation

In [15]:
window_spec=Window.partitionBy('userid').orderBy(col('injestion_timestamp').desc())

StatementMeta(, 7f236da4-c644-4002-809a-9f04bdba72ec, 31, Finished, Available, Finished, False)

In [16]:
silver_latest_df=(silver_df.withColumn('rn',row_number().over(window_spec)).filter(col('rn')==1).drop('rn'))

StatementMeta(, 7f236da4-c644-4002-809a-9f04bdba72ec, 32, Finished, Available, Finished, False)

In [18]:
print('silver total record:',silver_df.count())
print('latest customers :',silver_latest_df.count())
print('Unique user ids:',silver_latest_df.select('userid').distinct().count())


StatementMeta(, 7f236da4-c644-4002-809a-9f04bdba72ec, 34, Finished, Available, Finished, False)

silver total record: 506378
latest customers : 503344
Unique user ids: 503344


In [6]:
print('silver total record:',silver_df.count())
print('latest customers :',silver_latest_df.count())
print('Unique user ids:',silver_latest_df.select('userid').distinct().count())


StatementMeta(, 7f236da4-c644-4002-809a-9f04bdba72ec, 15, Finished, Available, Finished, False)

silver total record: 443356
latest customers : 440868
Unique user ids: 440868


### Build Gold Customer Dataset

In [ ]:
gold_customer_df=silver_latest_df.select(
    col('userid').alias('user_id'),
    col('Gender').alias('gender'),
    col('Title').alias('title'),
    col("First").alias('first_name'),
    col('Last').alias('last_name'),
    concat_ws(' ',trim(col('first')),trim(col('last'))).alias('full_name'),
    col('Email').alias('email'),
    col('City').alias('city'),
    col('State').alias('state'),
    col('Country').alias('country'),
    col('Postcode').alias('postcode'),
    col("Latitude").cast('double').alias('latitude'),
    col("Longitude").cast('double').alias('longitude'),
    col('NAT').alias('nationality'),
    col('Age').alias('age'),
    col('Birth_Date').alias('birth_date'),
    col('Registered_Date').alias('registered_date'),
    col('Registered_Age').cast('int').alias('registered_age'),
    col('Phone_Number').alias('phone_number'),
    col('Cell_Number').alias('cell_number'),
    col('Injestion_Timestamp').alias("injection_timestamp"),
    col('Processing_Timestamp').alias('processing_timestamp')

)

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 23, Finished, Available, Finished, False)

In [ ]:
print('Gold customer record:',gold_customer_df.count())
print('unique usser:',gold_customer_df.select('user_id').distinct().count())

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 24, Finished, Available, Finished, False)

Gold customer record: 423219


unique usser: 423219


In [ ]:
display(gold_customer_df.groupBy('user_id').count().filter(col('count')>1))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fa78cc1c-853d-4e21-b965-d908b1cd996d)

### Display sample 10 records

In [ ]:
display(gold_customer_df.limit(10))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, df5b24b6-4199-4df3-8546-2027cdca2df7)

### Create Customer Segmentation

In [ ]:
gold_segment_df=(
    gold_customer_df
    .withColumn("age_group",
        when(col('age').isNull(),'Unknown').
        when(col('age')<18,'Under 18').
        when(col('age').between(18,25),'18-25').
        when(col('age').between(26,35),'26-35').
        when(col('age').between(36,50),'36-50').
        when(col('age')>50,'51+').otherwise('Unknown')
        )
)

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 27, Finished, Available, Finished, False)

### checks for the segmented customers

In [ ]:
display(gold_segment_df.select('user_id','gender','age','age_group','country','nationality'))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 54dbbef3-191a-4365-913e-be6052975ebc)

### Demographic Analytics

In [ ]:
gold_demographics_df=(
    gold_segment_df.groupBy(
        'gender',
        'age_group',
        'nationality'
    ).count()
    .withColumnRenamed('count','customer_count')
)

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 29, Finished, Available, Finished, False)

### check for Demographics

In [ ]:
display(gold_demographics_df.orderBy(col('customer_count').desc()))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 840791e4-fde2-4cca-bf05-6ee817350e54)

### Geographic Market Analysis

In [ ]:
gold_geography_df=(
    gold_segment_df.groupBy(
        'country',
        'state',
        'city'
    ).count()
    .withColumnRenamed('count','customer_count')
)

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 31, Finished, Available, Finished, False)

### check for geographic counts

In [ ]:
display(gold_geography_df.orderBy(col('customer_count').desc()))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 836ab8e8-80ae-4631-b987-07a8c870bbb4)

### Data Quality Gold

In [ ]:
gold_quality_df=(
    gold_customer_df
    .withColumn(
        "profile_completeness_score",
        (
            when(col('first_name').isNotNull(),1).otherwise(0)
            +
            when(col('last_name').isNotNull(),1).otherwise(0)
            +
            when(col('email').isNotNull(),1).otherwise(0)
            +
            when(col('phone_number').isNotNull(),1).otherwise(0)
            +
            when(col('country').isNotNull(),1).otherwise(0)
            +
            when(col('city').isNotNull(),1).otherwise(0)
            +
            when(col('age').isNotNull(),1).otherwise(0)
        )
    )
)

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 33, Finished, Available, Finished, False)

### Check for Data completeness

In [ ]:
display(gold_quality_df.select(
    'user_id',
    'full_name',
    'email',
    'phone_number',
    'country',
    'age',
    'profile_completeness_score'
))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5422f2e7-54df-4138-ad22-06003dce25bb)

### Pipeline Monitoring

In [ ]:
gold_pipeline_df=(
    gold_customer_df
    .withColumn('processing_latency_seconds',
    unix_timestamp('processing_timestamp')
    -
    unix_timestamp('injection_timestamp'))
)

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 35, Finished, Available, Finished, False)

### check for Pipeline Monitoring

In [ ]:
display(gold_pipeline_df.select(
    'user_id',
    'injection_timestamp',
    'processing_timestamp',
    'processing_latency_seconds'
))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 514c0993-d940-4642-b203-4220ea1e6cb2)

### Load gold customers data to gold layer

In [ ]:
gold_customer_df.write.format('delta').mode('overwrite').saveAsTable('Gold.gold_customers')

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 37, Finished, Available, Finished, False)

In [ ]:
gold_customer_table=spark.table('gold.gold_customers')

print('gold table record:',gold_customer_table.count())

print('gold table unique record:',gold_customer_table.select('user_id').distinct().count())

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 38, Finished, Available, Finished, False)

gold table record: 423219


gold table unique record: 423219


In [ ]:
display(gold_customer_table.groupBy('user_id').count().filter(col('count')>1))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a42c94a4-7b4b-46f0-b9fd-758043b836f9)

#### validate the data load in gold tables

In [ ]:
display(spark.table('gold.gold_customers').limit(10))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fcadce4c-5abe-4dee-98df-193f174561cd)

In [20]:
gold_demographics_df.write.format('delta').mode('overwrite').saveAsTable('Gold.gold_demographics')

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 41, Finished, Available, Finished, False)

In [21]:
display(spark.table('gold.gold_demographics').limit(10))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ae5af085-291d-4d49-9874-e344ef062109)

In [22]:
gold_geography_df.write.format('delta').mode('overwrite').saveAsTable("Gold.gold_geography")

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 44, Finished, Available, Finished, False)

In [24]:
display(spark.table('gold.gold_geography').limit(10))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 782e8810-c80c-4f75-9b08-c0d2f3a834ae)

In [25]:
gold_quality_final_df=(
    gold_quality_df.withColumn(
        "profile_quality",
        when(col('profile_completeness_score')==7,"Complete").
        when(col('profile_completeness_score')>=5,"Mostly Complete").
        when(col('profile_completeness_score')>=3,"Partially Complete").otherwise('Poor')
    )
)

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 46, Finished, Available, Finished, False)

In [27]:
display(gold_quality_final_df.select(
    'user_id',
    'full_name',
    'profile_completeness_score',
    'profile_quality'
))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b8c5476a-f1c6-4015-a5f6-998b12a1f3b6)

In [28]:
gold_quality_final_df.write.format('delta').mode('overwrite').saveAsTable("Gold.gold_data_quality")

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 48, Finished, Available, Finished, False)

In [30]:
display(spark.table('gold.gold_data_quality'))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 664e82ba-947f-4f80-931c-fda41ebc5bbd)

In [31]:
gold_pipeline_df.write.format('delta').mode('overwrite').saveAsTable('Gold.gold_pipeline_metrics')

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 50, Finished, Available, Finished, False)

In [32]:
display(spark.table('gold.gold_pipeline_metrics'))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ecb62b02-98f0-4af3-85c8-9ca4b48f9d5b)

In [35]:
spark.sql('Show tables in Gold').show(truncate=False)

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 52, Finished, Available, Finished, False)

+---------------------+---------------------+-----------+
|namespace            |tableName            |isTemporary|
+---------------------+---------------------+-----------+
|Kiran.RT_Project.Gold|gold_customers       |false      |
|Kiran.RT_Project.Gold|gold_data_quality    |false      |
|Kiran.RT_Project.Gold|gold_demographics    |false      |
|Kiran.RT_Project.Gold|gold_geography       |false      |
|Kiran.RT_Project.Gold|gold_pipeline_metrics|false      |
+---------------------+---------------------+-----------+



In [ ]:
display(silver_df.count())

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 53, Finished, Available, Finished, False)

425615

In [ ]:
spark.sql('show tables in silver').show(truncate=False)

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 54, Finished, Available, Finished, False)

+-----------------------+---------------+-----------+
|namespace              |tableName      |isTemporary|
+-----------------------+---------------+-----------+
|Kiran.RT_Project.silver|api_silver_data|false      |
+-----------------------+---------------+-----------+



In [ ]:
gold_tables=['gold.gold_data_quality','gold.gold_demographics','gold.gold_geography','gold.gold_pipeline_metrics']
for table in gold_tables:
    df=spark.table(f"{table}")
    print(f"{table}:",{df.count()})

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 55, Finished, Available, Finished, False)

gold.gold_data_quality: {423219}


gold.gold_demographics: {168}
gold.gold_geography: {150269}


gold.gold_pipeline_metrics: {423219}


In [1]:
display(spark.sql("select count('*') from gold.gold_customers"))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 157, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37948661-3cb4-45e4-b97e-2b790cc2a920)

In [2]:
display(spark.sql("select * from gold.gold_customers limit(10)"))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 158, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e16155dc-acae-4d55-b454-1697f2b81f4b)

In [4]:
display(spark.sql("""SELECT count(*) total,count(distinct user_id) unique_users from gold.gold_customers"""))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 160, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 19ae9ff4-144f-4a35-a65f-74ca261c1dbe)

In [16]:
display(spark.sql("""
select user_id,first_name,last_name from gold.gold_customers where user_id='0053148a-55b0-4745-91b8-958883bc4df9'"""))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 172, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 381637e8-8948-4670-b49b-70697b90b5d7)

In [13]:
df_sil=spark.table('silver.api_silver_data')

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 169, Finished, Available, Finished, False)

In [12]:
display(df_sil.filter(col('userid')=='0053148a-55b0-4745-91b8-958883bc4df9'))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 168, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ad49d8ec-5519-41af-a30e-b176bbc8c12c)

In [21]:
df=spark.table('gold.gold_customers')
display(df.count())
display(df.select(countDistinct('user_id')))

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 177, Finished, Available, Finished, False)

434065

SynapseWidget(Synapse.DataFrame, d89a3b77-21b3-434e-9de5-03cfc2dcdf48)

In [26]:
missing=df_sil.join(df,df.user_id==df_sil.userid,'left_anti')
print(missing.count())

StatementMeta(, 6474e44c-50a0-407c-9787-343ba4bcd384, 196, Finished, Available, Finished, False)

0
